In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Preprocessing for XGBoost

def preprocess_data(data, brute_force_threshold_seconds=60, dos_threshold=100):
    # Add URL-based features
    data['url_length'] = data['request'].str.len()  # Length of the URL
    data['num_params'] = data['request'].str.count('&') + 1  # Number of parameters in the URL

    # Add attack pattern features
    sql_injection_pattern = r"UNION|SELECT|INSERT|DELETE|UPDATE|DROP|CREATE|ALTER|--|/\*.*?\*/|%27"
    xss_pattern = r"<script>|%3Cscript%3E|javascript:|onload|eval|src=['\"]javascript:"
    dir_traversal_pattern = r"\..\/|\..\\|%2e%2e%2f|%2e%2e%5c|etc/passwd|boot.ini"

    data['contains_sql_keywords'] = data['request'].str.contains(sql_injection_pattern, case=False, na=False).astype(int)
    data['contains_xss_keywords'] = data['request'].str.contains(xss_pattern, case=False, na=False).astype(int)
    data['contains_dir_traversal'] = data['request'].str.contains(dir_traversal_pattern, case=False, na=False).astype(int)

    # Add Brute Force feature (based on time difference between requests)
    data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
    data.sort_values(by=['client_ip', 'timestamp'], inplace=True)
    data['time_diff'] = data.groupby('client_ip')['timestamp'].diff().dt.total_seconds()
    data['is_brute_force'] = data['time_diff'] <= brute_force_threshold_seconds

    # Add DoS feature (based on request rate)
    data['request_rate'] = data.groupby('client_ip')['client_ip'].transform('count')
    data['is_dos'] = data['request_rate'] > dos_threshold

    # Encode categorical features
    label_encoders = {}
    categorical_columns = ['method', 'protocol', 'user_agent']
    for col in categorical_columns:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col].astype(str))
        label_encoders[col] = le

    # Drop non-numeric or non-essential columns
    data = data.drop(columns=['client_ip', 'request', 'referrer', 'timestamp'], errors='ignore')

    # Rename corrected_label to label safely
    if 'corrected_label' in data.columns:
        data = data.rename(columns={'corrected_label': 'label'})
    elif 'label' in data.columns and 'corrected_label' in data.columns:
        data = data.drop(columns=['label'], errors='ignore').rename(columns={'corrected_label': 'label'})

    # Encode target labels
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    return data, label_encoder

# Load and preprocess the dataset
def load_and_preprocess(file_path):
    data = pd.read_csv(file_path)
    processed_data, label_encoder = preprocess_data(data)
    return processed_data, label_encoder

# Train and evaluate XGBoost with hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

def train_and_evaluate_xgboost(data, label_encoder, test_size=0.2, random_state=42):
    # Split into features and target
    X = data.drop(columns=['label'], errors='ignore')
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

    # Define XGBoost model and parameters for RandomizedSearchCV
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=random_state)
    param_dist = {
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'n_estimators': [50, 100, 200],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }

    randomized_search = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_dist, n_iter=20,
                                           scoring='accuracy', cv=3, verbose=1, n_jobs=1, random_state=random_state)
    randomized_search.fit(X_train, y_train)

    # Best model from random search
    best_model = randomized_search.best_estimator_

    # Evaluate the best model
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    classification_rep = classification_report(y_test, y_pred, target_names=label_encoder.classes_)

    return best_model, accuracy, classification_rep, randomized_search.best_params_

# Main function to execute the workflow
input_file = './dataset/logs_last_modified.csv'  # Adjust path as needed

# Load and preprocess dataset
data, label_encoder = load_and_preprocess(input_file)

# Train and evaluate the model with hyperparameter tuning
model, accuracy, report, best_params = train_and_evaluate_xgboost(data, label_encoder)

# Print results
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", report)
print("\nBest Parameters from RandomizedSearchCV:\n", best_params)


Fitting 3 folds for each of 20 candidates, totalling 60 fits


d:\Anaconda\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:33:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Anaconda\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:33:06] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Anaconda\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:33:08] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Anaconda\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:33:10] WARNING

Accuracy: 99.99%

Classification Report:
                      precision    recall  f1-score   support

        Brute Force       1.00      1.00      1.00     37088
Directory Traversal       1.00      1.00      1.00       123
             Normal       1.00      1.00      1.00      2764
      SQL Injection       0.96      0.81      0.88        27

           accuracy                           1.00     40002
          macro avg       0.99      0.95      0.97     40002
       weighted avg       1.00      1.00      1.00     40002


Best Parameters from RandomizedSearchCV:
 {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 1.0}


In [3]:
import pandas as pd

# Load the uploaded CSV file to inspect its contents
file_path = './dataset/logs_last_modified.csv'
df = pd.read_csv(file_path)

# Display the first few rows and the column names to understand its structure
df.head(), df.columns


(    client_ip                  timestamp method  \
 0   10.19.0.3  2023-04-14 10:10:47+07:00    GET   
 1   10.19.0.3  2023-04-14 16:45:17+07:00    GET   
 2  10.200.7.2  2023-04-12 07:49:00+07:00    GET   
 3  10.200.7.2  2023-04-12 07:49:03+07:00    GET   
 4  10.200.7.2  2023-04-12 07:59:01+07:00    GET   
 
                                              request  protocol  status  \
 0  /yayasan-pendidikan-telkom-siap-hadirkan-insti...  HTTP/1.0     200   
 1                                                  /  HTTP/1.0     200   
 2                                                  /  HTTP/1.0     200   
 3                                       /favicon.ico  HTTP/1.0     302   
 4                                                  /  HTTP/1.0     200   
 
    response_size                     referrer  \
 0         182649      https://www.google.com/   
 1         258016                            -   
 2         257487      https://www.google.com/   
 3              0  https://ittelko

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Update target column to 'label'
target_column = 'label'

# Encoding categorical features
label_encoders = {}
for column in df.select_dtypes(include=['object']).columns:
    if column != target_column:  # Avoid encoding the target column here
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

# Encoding the target column
label_encoder_target = LabelEncoder()
df[target_column] = label_encoder_target.fit_transform(df[target_column])

# Feature engineering: if there is a time-related column, create new features
if 'timestamp' in df.columns:
    df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
    df['day_of_week'] = pd.to_datetime(df['timestamp']).dt.dayofweek
    df['month'] = pd.to_datetime(df['timestamp']).dt.month
    df.drop('timestamp', axis=1, inplace=True)

# Splitting the dataset into features and target
X = df.drop(target_column, axis=1)
y = df[target_column]

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizing the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Training the XGBoost classifier
model = XGBClassifier()
model.fit(X_train, y_train)

# Predicting and evaluating the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

# Print accuracy
print(f"Accuracy: {accuracy}\n")

# Print classification report
print("Classification Report:")
print(classification_rep)


Accuracy: 0.9720513974301285

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     37154
           1       1.00      1.00      1.00       125
           2       0.81      0.76      0.79      2692
           3       1.00      0.84      0.91        31

    accuracy                           0.97     40002
   macro avg       0.95      0.90      0.92     40002
weighted avg       0.97      0.97      0.97     40002



In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Update target column to 'label'
target_column = 'label'

# Encoding categorical features
label_encoders = {}
for column in df.select_dtypes(include=['object']).columns:
    if column != target_column:  # Avoid encoding the target column here
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

# Encoding the target column
label_encoder_target = LabelEncoder()
df[target_column] = label_encoder_target.fit_transform(df[target_column])

# Feature engineering: if there is a time-related column, create new features
if 'timestamp' in df.columns:
    df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
    df['day_of_week'] = pd.to_datetime(df['timestamp']).dt.dayofweek
    df['month'] = pd.to_datetime(df['timestamp']).dt.month
    df.drop('timestamp', axis=1, inplace=True)

# Splitting the dataset into features and target
X = df.drop(target_column, axis=1)
y = df[target_column]

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizing the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Training the XGBoost classifier
model = XGBClassifier()
model.fit(X_train, y_train)

# Predicting and evaluating the model
y_pred = model.predict(X_test)

# Decode the target names from LabelEncoder
y_test_decoded = label_encoder_target.inverse_transform(y_test)
y_pred_decoded = label_encoder_target.inverse_transform(y_pred)
target_names = list(map(str, label_encoder_target.classes_))  # Ensure target names are strings

classification_rep = classification_report(y_test_decoded, y_pred_decoded, target_names=target_names)

# Print accuracy
accuracy = accuracy_score(y_test_decoded, y_pred_decoded)
print(f"Accuracy: {accuracy}\n")

# Print classification report
print("Classification Report:")
print(classification_rep)



Accuracy: 0.9720513974301285

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     37154
           1       1.00      1.00      1.00       125
           2       0.81      0.76      0.79      2692
           3       1.00      0.84      0.91        31

    accuracy                           0.97     40002
   macro avg       0.95      0.90      0.92     40002
weighted avg       0.97      0.97      0.97     40002

